# Mendelian Genetics - PART 3 (FINAL)

## Chi-Square Test & Gene Interactions

**This is Part 3 of 3**

---

## Part 3 Contents:

1. Chi-square statistical test
2. Gene interactions explorer
3. Summary and practice problems

---

# 🎯 Chi-Square Test

## The Question

You observe: 95 dominant, 25 recessive

**Is this close enough to 3:1?**

**Answer:** Chi-square test!

### Formula:

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

- O = Observed
- E = Expected

In [ ]:
# Chi-Square Calculator

@interact(
    obs_dom=IntSlider(min=50, max=150, value=95, description='Obs Dominant:'),
    obs_rec=IntSlider(min=10, max=50, value=25, description='Obs Recessive:'),
    alpha=Dropdown(options=[0.05, 0.01], value=0.05, description='α:')
)
def chi_square_test(obs_dom, obs_rec, alpha):
    total = obs_dom + obs_rec
    
    # Expected for 3:1
    exp_dom = total * 0.75
    exp_rec = total * 0.25
    
    observed = [obs_dom, obs_rec]
    expected = [exp_dom, exp_rec]
    
    # Calculate chi-square
    chi_sq = sum((o - e)**2 / e for o, e in zip(observed, expected))
    
    # Critical value (df=1)
    critical = chi2.ppf(1 - alpha, 1)
    p_value = 1 - chi2.cdf(chi_sq, 1)
    reject = chi_sq > critical
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Bar chart
    ax = axes[0]
    x = [0, 1]
    width = 0.35
    
    ax.bar([i-width/2 for i in x], observed, width, label='Observed', 
           color='steelblue', alpha=0.8, edgecolor='black', linewidth=2)
    ax.bar([i+width/2 for i in x], expected, width, label='Expected', 
           color='coral', alpha=0.8, edgecolor='black', linewidth=2)
    
    ax.set_ylabel('Count', fontsize=13, fontweight='bold')
    ax.set_title('Observed vs Expected', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Dominant', 'Recessive'])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Chi-square distribution
    ax2 = axes[1]
    x_range = np.linspace(0, max(critical*2, chi_sq*1.5), 200)
    y_range = chi2.pdf(x_range, 1)
    
    ax2.plot(x_range, y_range, 'b-', linewidth=2, label='χ² dist (df=1)')
    ax2.fill_between(x_range[x_range >= critical], 0, 
                     chi2.pdf(x_range[x_range >= critical], 1),
                     color='red', alpha=0.3, label=f'Reject region (α={alpha})')
    
    ax2.axvline(chi_sq, color='green', linewidth=3, linestyle='--', 
               label=f'χ² = {chi_sq:.3f}')
    ax2.axvline(critical, color='red', linewidth=2, linestyle=':', 
               label=f'Critical = {critical:.3f}')
    
    ax2.set_xlabel('χ²', fontsize=12, fontweight='bold')
    ax2.set_title('Chi-Square Distribution', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # Decision
    ax3 = axes[2]
    ax3.axis('off')
    
    color = 'red' if reject else 'green'
    decision = 'REJECT' if reject else 'ACCEPT'
    
    text = f"""Chi-Square Test

χ² = {chi_sq:.3f}
Critical = {critical:.3f}
p-value = {p_value:.4f}

Decision: {decision} H₀

{"Data does NOT fit 3:1" if reject else "Data fits 3:1"}
"""
    
    ax3.text(0.5, 0.5, text, ha='center', va='center', 
            transform=ax3.transAxes, fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.3, linewidth=3),
            family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    # Print
    print("\n🎯 Chi-Square Test:")
    print("═" * 60)
    print(f"Hypothesis: 3:1 ratio")
    print(f"α = {alpha}, df = 1")
    print(f"\nχ² = {chi_sq:.4f}")
    print(f"Critical = {critical:.4f}")
    print(f"p-value = {p_value:.4f}")
    print(f"\nDecision: {'REJECT' if reject else 'ACCEPT'} hypothesis")
    print("═" * 60)

---

# 🔄 Gene Interactions

## Beyond Simple Dominance

Not all genes follow 3:1 or 9:3:3:1!

### Types:

1. **Incomplete Dominance** - Intermediate phenotype (1:2:1)
2. **Codominance** - Both expressed (1:2:1)
3. **Epistasis** - One gene masks another (modified ratios)

In [ ]:
# Gene Interaction Explorer

@interact(
    interaction=Dropdown(
        options=['Complete Dominance (3:1)', 
                'Incomplete Dominance (1:2:1)',
                'Epistasis 9:3:4'],
        value='Complete Dominance (3:1)',
        description='Type:'),
    n=IntSlider(min=100, max=1000, step=100, value=400, description='Sample:')
)
def gene_interactions(interaction, n):
    
    if 'Complete' in interaction:
        phenotypes = ['Dominant', 'Recessive']
        props = [0.75, 0.25]
        colors = ['purple', 'white']
        info = "AA and Aa → dominant\naa → recessive\n3:1 ratio"
    
    elif 'Incomplete' in interaction:
        phenotypes = ['Red', 'Pink', 'White']
        props = [0.25, 0.50, 0.25]
        colors = ['red', 'pink', 'white']
        info = "RR = red\nRr = pink (intermediate)\nrr = white\n1:2:1 ratio"
    
    else:  # Epistasis
        phenotypes = ['Black', 'Brown', 'Yellow']
        props = [9/16, 3/16, 4/16]
        colors = ['black', 'brown', 'yellow']
        info = "E_B_ = black (9/16)\nE_bb = brown (3/16)\nee__ = yellow (4/16)\n9:3:4 ratio"
    
    # Simulate
    observed = np.random.multinomial(n, props)
    expected = [n * p for p in props]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    ax = axes[0]
    x = np.arange(len(phenotypes))
    width = 0.35
    
    ax.bar(x - width/2, observed, width, label='Observed', 
           color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax.bar(x + width/2, expected, width, label='Expected', 
           color='gray', alpha=0.5, edgecolor='black', linewidth=2)
    
    ax.set_ylabel('Count', fontsize=13, fontweight='bold')
    ax.set_title(f'{interaction}\n(n={n})', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(phenotypes)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    for i, val in enumerate(observed):
        ax.text(i - width/2, val + max(observed)*0.02, f'{val}', 
               ha='center', fontweight='bold')
    
    # Info
    ax2 = axes[1]
    ax2.axis('off')
    ax2.text(0.1, 0.5, info, transform=ax2.transAxes, 
            fontsize=12, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
            family='monospace')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n🔄 {interaction}")
    print("═" * 60)
    print(f"Expected ratio: {':'.join([str(int(p*16)) for p in props])}")
    print("\n💡 Different interactions → Different ratios!")
    print("═" * 60)

---

# 🎯 Summary

## What You've Learned:

### ✅ Part 1: Monohybrid Cross
- 3:1 ratio from Tt × Tt
- Mendel's Law of Segregation
- Binomial distribution

### ✅ Part 2: Probability & Dihybrid
- Product and sum rules
- 9:3:3:1 ratio from RrYy × RrYy
- Independent Assortment

### ✅ Part 3: Chi-Square & Interactions
- Statistical validation of ratios
- Incomplete dominance, codominance, epistasis
- Modified ratios

---

## Practice Problems

**Problem 1:** Cross Tt × Tt. In 200 offspring, how many short?
- **Answer:** 50

**Problem 2:** Cross TtPp × TtPp. What fraction tall purple?
- **Answer:** 9/16

**Problem 3:** Observed 142 wild, 48 mutant. Test for 3:1.
- **Answer:** χ² ≈ 0.004, Accept

---

## 🙏 Acknowledgments

**Developed by:**
- Susama Kar (Lecturer in Zoology)
- Dr. Alok Patel (Head, Zoology Dept)

**Institution:** Kuchinda College, Sambalpur University, Odisha

**Philosophy:** Pattern Hunters - "Uncertainty has predictable shapes"

**License:** CC BY 4.0

**Repository:** github.com/The-Pattern-Hunter/principles-of-genetics-interactive

---

## ✅ COMPLETE!

**You've completed all 3 parts of Mendelian Genetics!**

**Next steps:**
1. Save this merged notebook
2. Practice with the interactive widgets
3. Try the practice problems
4. Continue to linkage and mapping notebooks!